In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Data Loading and Cleaning ---
file_name = "Global_Climate_Change_Data_2020_2025.csv"
df = pd.read_csv(file_name)

# Rename Columns for easier access
new_columns = {
    'Avg_Temperature(°C)': 'Avg_Temperature_C',
    'CO2_Emissions(Mt)': 'CO2_Emissions_Mt',
    'Sea_Level_Rise(mm)': 'Sea_Level_Rise_mm'
}
df.rename(columns=new_columns, inplace=True)

print("--- Initial Data Check ---")
print(df.head())
print("\n")

# --- 2. Data Wrangling (Functions and Conditionals) ---

def handle_missing_data(data_frame, column, strategy='mean'):
    missing_count = data_frame[column].isnull().sum()
    print(f"Missing values in '{column}' before handling: {missing_count}")
    
    if data_frame[column].isnull().any():
        if strategy == 'mean':
            impute_value = data_frame[column].mean()
            data_frame[column].fillna(impute_value, inplace=True)
            print(f"Filled missing values in '{column}' with the mean: {impute_value:.2f}")
    return data_frame

df = handle_missing_data(df, 'CO2_Emissions_Mt', 'mean')
print("\n")

def categorize_risk(risk_index):
    if risk_index < 30:
        return 'Low'
    elif risk_index < 60:
        return 'Medium'
    else:
        return 'High'

df['Climate_Risk_Category'] = df['Climate_Risk_Index'].apply(categorize_risk)

df.to_csv('Global_Climate_Change_Data_Wrangled.csv', index=False)
print("Wrangled data saved to Global_Climate_Change_Data_Wrangled.csv\n")

# --- 3. NumPy Analysis ---
co2_emissions_np = df['CO2_Emissions_Mt'].to_numpy()
co2_mean = np.mean(co2_emissions_np)
co2_std = np.std(co2_emissions_np)

print("--- NumPy Analysis on CO2 Emissions ---")
print(f"Mean CO2 Emissions: {co2_mean:.2f} Mt")
print(f"Standard Deviation of CO2 Emissions: {co2_std:.2f} Mt\n")


# --- 4. Pandas Group Analysis ---
continent_avg_df = (
    df.groupby('Continent')
      .mean(numeric_only=True)    # compute means (Year will be averaged too)
      .drop(columns=['Year'])     # remove the averaged Year so that the years dont show as decimals
      .round(2)                   # round the calculated numeric results to 2 decimal places
      .reset_index()
)

continent_avg_df_sorted = continent_avg_df.sort_values(by='Climate_Risk_Index', ascending=False)

print("--- Average Climate Metrics by Continent (Sorted by Climate Risk Index) ---")
print(continent_avg_df_sorted)
print("\n")

# --- 5. Matplotlib Plot 1: CO₂ by Continent ---
plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(continent_avg_df_sorted)))
plt.bar(continent_avg_df_sorted['Continent'], continent_avg_df_sorted['CO2_Emissions_Mt'], color=colors)

plt.title('Average CO2 Emissions (Mt) by Continent (2020-2025)')
plt.xlabel('Continent')
plt.ylabel('Average CO2 Emissions (Mt)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--')
plt.tight_layout()
plt.savefig('avg_co2_emissions_by_continent_mpl_fixed.png')
plt.close()
print("Saved plot: avg_co2_emissions_by_continent_mpl_fixed.png")

# --- 6. Matplotlib Plot 2: Temperature Trends for Top 3 Continents ---
top_3_continents = continent_avg_df_sorted['Continent'].head(3).tolist()
temp_trend_df = df[df['Continent'].isin(top_3_continents)].groupby(['Year', 'Continent'])['Avg_Temperature_C'].mean().reset_index()

plt.figure(figsize=(12, 7))
colors = plt.cm.coolwarm(np.linspace(0, 1, len(top_3_continents)))

for continent, color in zip(top_3_continents, colors):
    subset = temp_trend_df[temp_trend_df['Continent'] == continent]
    plt.plot(subset['Year'], subset['Avg_Temperature_C'], marker='o', label=continent, color=color, linewidth=2)

plt.title(f'Average Temperature Trend ({", ".join(top_3_continents)})')
plt.xlabel('Year')
plt.ylabel('Average Temperature (°C)')
plt.legend(title='Continent')
plt.xticks(temp_trend_df['Year'].unique())
plt.grid(True)
plt.tight_layout()
plt.savefig('temp_trend_top_3_continents_mpl_fixed.png')
plt.close()
print("Saved plot: temp_trend_top_3_continents_mpl_fixed.png")




# ---------------------------------------------------------------------
# Correlation Visualization (Two Line Charts) -----
# ---------------------------------------------------------------------

# I made two line charts visualized, one holding the temperature change over the years, and another holding the CO2 emissions by year.
# When placed side by side, you can see the correlation on how they both increased and reduced over the years in a similar manner.
# Although they didn't move exactly in the same manner, there were strong similarities especially from 2021-2024

# 1) CO₂ per year
co2_per_year = df.groupby('Year')['CO2_Emissions_Mt'].sum().reset_index()

plt.figure(figsize=(10, 6))
plt.plot(co2_per_year['Year'], co2_per_year['CO2_Emissions_Mt'], marker='o', linewidth=2)
plt.title("Total CO2 Emissions per Year (2020–2025)")
plt.xlabel("Year")
plt.ylabel("CO2 Emissions (Mt)")
plt.grid(True)
plt.tight_layout()
plt.savefig("co2_emissions_by_year_line.png")
plt.close()

# 2) Temperature per year
temp_per_year = df.groupby('Year')['Avg_Temperature_C'].mean().reset_index()

plt.figure(figsize=(10, 6))
plt.plot(temp_per_year['Year'], temp_per_year['Avg_Temperature_C'], marker='o', linewidth=2)
plt.title("Average Temperature per Year (2020–2025)")
plt.xlabel("Year")
plt.ylabel("Avg Temperature (°C)")
plt.grid(True)
plt.tight_layout()
plt.savefig("temperature_by_year_line.png")
plt.close()

print("Saved plot: co2_emissions_by_year_line.png")
print("Saved plot: temperature_by_year_line.png")
print("\nLine charts created. The shapes of the plot can be compared side by side to evaluate correlation.")
print("The correlation of the CO2 emission and heat changes were mainly very visible in the years 2021 -2024.")

--- Initial Data Check ---
   Year      Continent Country  Avg_Temperature_C  CO2_Emissions_Mt  \
0  2021         Europe      UK               19.6            978.24   
1  2022           Asia   India               25.3            770.39   
2  2022           Asia   Japan               23.2            963.84   
3  2020  North America  Mexico               20.8            349.49   
4  2024         Africa   Egypt               33.1            475.82   

   Sea_Level_Rise_mm  Climate_Risk_Index  
0               3.57                  28  
1               1.47                  74  
2               3.09                  48  
3               3.81                  23  
4               3.35                  86  


Missing values in 'CO2_Emissions_Mt' before handling: 0


Wrangled data saved to Global_Climate_Change_Data_Wrangled.csv

--- NumPy Analysis on CO2 Emissions ---
Mean CO2 Emissions: 556.83 Mt
Standard Deviation of CO2 Emissions: 258.29 Mt

--- Average Climate Metrics by Continent (Sort

In [ ]:
# -------------------------------------------------------
# China % of Asia CO₂ (2020–2025), USA % of North America's CO₂ ---
# -------------------------------------------------------

asia_df = df[df['Continent'] == 'Asia']
china_df = df[df['Country'] == 'China']

asia_total = asia_df.groupby('Year')['CO2_Emissions_Mt'].sum().sum()
china_total = china_df.groupby('Year')['CO2_Emissions_Mt'].sum().sum()

china_percent = (china_total / asia_total) * 100

print("\n--- China CO2 Contribution to Asia (2020–2025) ---")
print(f"Total Asia CO2 (2020–2025): {asia_total:.2f} Mt")
print(f"Total China CO2 (2020–2025): {china_total:.2f} Mt")
print(f"China contributed {china_percent:.2f}% of Asia's total CO2 emissions.\n")


#USA % of North America's CO₂ ---
north_america_df = df[df['Continent'] == 'North America'] #I'm using this to locate 'North America' in the Continent column
usa_df = df[df['Country'] =='USA'] #I'm using this to locate 'USA' in the Country column

north_america_total = north_america_df.groupby('Year')['CO2_Emissions_Mt'].sum().sum()
usa_total = usa_df.groupby('Year')['CO2_Emissions_Mt'].sum().sum()

usa_percent = (usa_total / north_america_total) * 100
print("\n--- USA CO2 Contribution to North America (2020–2025) ---")
print(f"Total North America's CO2 (2020–2025): {north_america_total:.2f} Mt")
print(f"Total USA CO2 (2020–2025): {usa_total:.2f} Mt")
print(f"The United States contributed {usa_percent:.2f}% of North America's total CO2 emissions.\n")



--- China CO2 Contribution to Asia (2020–2025) ---
Total Asia CO2 (2020–2025): 116196.40 Mt
Total China CO2 (2020–2025): 17250.01 Mt
China contributed 14.85% of Asia's total CO2 emissions.


--- USA CO2 Contribution to North America (2020–2025) ---
Total North America's CO2 (2020–2025): 111476.11 Mt
Total USA CO2 (2020–2025): 33615.15 Mt
The United States contributed 30.15% of North America's total CO2 emissions.



In [1]:
import pandas as pd
import time
import psutil
import os

print("\n--- 1. Small Data on PANDAS (Climate Change) ---")

file = "Global_Climate_Change_Data_2020_2025.csv"

# ------------------------------------------
# 1. MEMORY BEFORE LOADING DATA
# ------------------------------------------
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 ** 2)  # MB
print(f"Memory Before Load: {mem_before:.2f} MB")

# ------------------------------------------
# 2. LOADING DATA
# ------------------------------------------
start_load = time.time()

df_pd = pd.read_csv(file)

df_pd = df_pd.rename(columns={
    "Avg_Temperature(°C)": "Avg_Temperature_C",
    "CO2_Emissions(Mt)": "CO2_Emissions_Mt"
})

end_load = time.time()

# ------------------------------------------
# 3. GROUP OPERATION
# ------------------------------------------
start_group = time.time()

aggregated_pd = df_pd.groupby("Continent")["CO2_Emissions_Mt"].sum()

end_group = time.time()

# ------------------------------------------
# 4. MEMORY AFTER EXECUTION
# ------------------------------------------
mem_after = process.memory_info().rss / (1024 ** 2)  # MB
mem_used = mem_after - mem_before

print(f"Memory After Load & Aggregation: {mem_after:.2f} MB")
print(f"Memory Used: {mem_used:.2f} MB")

# ------------------------------------------
# 5. TIMING OUTPUT
# ------------------------------------------
print(f"Pandas Load Time: {end_load - start_load:.6f} sec")
print(f"Pandas Aggregation Time: {end_group - start_group:.6f} sec")
print(f"Pandas Total Time: {(end_group - start_load):.6f} sec")

print("------------------------------------------")



--- 1. Small Data on PANDAS (Climate Change) ---
Memory Before Load: 118.14 MB
Memory After Load & Aggregation: 119.11 MB
Memory Used: 0.97 MB
Pandas Load Time: 0.006760 sec
Pandas Aggregation Time: 0.001303 sec
Pandas Total Time: 0.008094 sec
------------------------------------------


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, col
import time
import psutil
import os

print("\n--- 2. Small Data on PYSPARK (Climate Change) ---")

file = "Global_Climate_Change_Data_2020_2025.csv"

# ------------------------------------------
# 1. MEMORY BEFORE LOADING DATA
# ------------------------------------------
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 ** 2)  # MB
print(f"Memory Before Load: {mem_before:.2f} MB")

# ------------------------------------------
# 2. SPARK INITIALIZATION AND LOADING
# ------------------------------------------
spark_start = time.time()

spark = (
    SparkSession.builder
        .master("local[*]")
        .appName("SmallDataSparkTest")
        .config("spark.driver.memory", "1g")
        .config("spark.jars.ivy", "NONE")
        .getOrCreate()
)

df_spark = spark.read.csv(file, header=True, inferSchema=True)

# Rename to remove parentheses from schema
df_spark = df_spark.withColumnRenamed("CO2_Emissions(Mt)", "CO2_Emissions_Mt")

load_done = time.time()

# ------------------------------------------
# 3. GROUP OPERATION (CO2 SUM)
# ------------------------------------------
result_spark = (
    df_spark.groupBy("Continent")
            .agg(sum(col("CO2_Emissions_Mt")).alias("Total_CO2_Emissions"))
)

result_spark.collect()  # execution trigger

spark_end = time.time()

# ------------------------------------------
# 4. MEMORY AFTER EXECUTION
# ------------------------------------------
mem_after = process.memory_info().rss / (1024 ** 2)  # MB
mem_used = mem_after - mem_before

print(f"Memory After Load & Aggregation: {mem_after:.2f} MB")
print(f"Memory Used: {mem_used:.2f} MB")

# ------------------------------------------
# 5. TIMING RESULTS
# ------------------------------------------
print(f"Spark Load Time: {load_done - spark_start:.6f} sec")
print(f"Spark Aggregation Time: {spark_end - load_done:.6f} sec")
print(f"Spark Total Time: {spark_end - spark_start:.6f} sec")

spark.stop()
print("------------------------------------------")



--- 2. Small Data on PYSPARK (Climate Change) ---
Memory Before Load: 2057.40 MB
Memory After Load & Aggregation: 2057.61 MB
Memory Used: 0.20 MB
Spark Load Time: 3.359945 sec
Spark Aggregation Time: 0.346772 sec
Spark Total Time: 3.706717 sec
------------------------------------------


In [1]:
import pandas as pd
import time
import psutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, col

# Obtain the current system process for memory measurement
process = psutil.Process()
file = "yellow_tripdata_2016-01.csv"

print("--- 3. Large Data on PANDAS (Taxi Data) ---")

# -----------------------
# 1. PANDAS TEST
# -----------------------
print("\n--- PANDAS TEST ---")

# Record memory usage prior to dataset loading
mem_before_load = process.memory_info().rss / (1024 * 1024)
start_pandas_load = time.time()

# Load the dataset fully into memory using Pandas
df_pd = pd.read_csv(file)

# Record time and memory after loading completes
end_pandas_load = time.time()
mem_after_load = process.memory_info().rss / (1024 * 1024)

# Begin execution timing for the aggregation operation
start_pandas_group = time.time()

# Group the data by VendorID and compute the total trip distance
pd_result = df_pd.groupby("VendorID")["trip_distance"].sum()

# Record time and memory after aggregation completes
end_pandas_group = time.time()
mem_after_group = process.memory_info().rss / (1024 * 1024)

# Display execution time and memory usage findings for Pandas
print(f"Pandas Load Time: {end_pandas_load - start_pandas_load:.2f} sec")
print(f"Pandas GroupBy Time: {end_pandas_group - start_pandas_group:.2f} sec")
print(f"Pandas Total Time: {(end_pandas_group - start_pandas_load):.2f} sec")
print(f"Pandas Memory Before Load: {mem_before_load:.2f} MB")
print(f"Pandas Memory After Load: {mem_after_load:.2f} MB")
print(f"Pandas Memory After Group: {mem_after_group:.2f} MB")


# -----------------------
# 2. PYSPARK TEST
# -----------------------
print("\n--- PYSPARK TEST ---")

# Start timing for Spark environment initialization and load
spark_start = time.time()

# Create the Spark session to enable distributed data processing
spark = (
    SparkSession.builder
    .appName("Benchmark")
    .master("local[*]")  # Utilize all local CPU cores
    .getOrCreate()
)

# Record memory before Spark data loading
mem_before_spark_load = process.memory_info().rss / (1024 * 1024)

# Load the same dataset into Spark, which reads data in partitions
df_spark = spark.read.csv(file, header=True, inferSchema=True)

# Record time and memory after Spark completes data reading
load_done = time.time()
mem_after_spark_load = process.memory_info().rss / (1024 * 1024)

# Apply aggregation: group by VendorID and compute the total distance
spark_result = (
    df_spark.groupBy(col("VendorID"))
    .agg(sum("trip_distance").alias("total_distance"))
)

# Trigger actual computation since Spark uses lazy execution
spark_result.collect()

# Record final time and memory usage after transformation completes
spark_end = time.time()
mem_after_spark_group = process.memory_info().rss / (1024 * 1024)

# Display execution time and memory usage findings for Spark
print(f"Spark Load Time: {load_done - spark_start:.2f} sec")
print(f"Spark GroupBy Time: {spark_end - load_done:.2f} sec")
print(f"Spark Total Time: {(spark_end - spark_start):.2f} sec")
print(f"Spark Memory Before Load: {mem_before_spark_load:.2f} MB")
print(f"Spark Memory After Load: {mem_after_spark_load:.2f} MB")
print(f"Spark Memory After Group: {mem_after_spark_group:.2f} MB")

# Terminate Spark session to release allocated resources
spark.stop()


--- 3. Large Data on PANDAS (Taxi Data) ---

--- PANDAS TEST ---
Pandas Load Time: 73.96 sec
Pandas GroupBy Time: 0.23 sec
Pandas Total Time: 74.20 sec
Pandas Memory Before Load: 126.64 MB
Pandas Memory After Load: 2039.07 MB
Pandas Memory After Group: 2041.84 MB

--- PYSPARK TEST ---
Spark Load Time: 62.79 sec
Spark GroupBy Time: 9.70 sec
Spark Total Time: 72.49 sec
Spark Memory Before Load: 2053.41 MB
Spark Memory After Load: 2055.86 MB
Spark Memory After Group: 2056.67 MB


In [ ]:
import pandas as pd
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, col

# Name of the large dataset we want to benchmark
file = "itineraries.csv"

print("--- Large Data Benchmark: Pandas vs PySpark (Itineraries Dataset) ---")


# ---------------------------------------------------------
# PYSPARK TEST
# ---------------------------------------------------------
print("\n--- PYSPARK TEST ---")

# Start a timer to measure how long Spark takes
spark_start = time.time()

# Create a SparkSession (this is the main entry point to PySpark)
# .master("local[*]") = use all available CPU cores on this machine
spark = (
    SparkSession.builder
    .appName("ItinerariesBenchmark")  # Name of the Spark job
    .master("local[*]")               # Run using local machine parallelism
    .getOrCreate()
)

# Load the CSV into a Spark DataFrame
# header=True → first row contains column names
# inferSchema=True → Spark automatically detects data types
df_spark = spark.read.csv(file, header=True, inferSchema=True)

# Mark the time right after loading finishes
load_done = time.time()

# Perform aggregation:
# Group the data by "startingAirport" and sum the "totalFare"
spark_result = (
    df_spark.groupBy(col("startingAirport"))   # Group rows by airport
    .agg(sum("totalFare").alias("total_fare")) # Calculate total fare per airport
)

# Trigger real computation:
# Spark uses lazy execution, so nothing actually runs until an action is called.
# .collect() forces Spark to execute the groupBy + sum operations.
spark_result.collect()

# Timestamp after Spark finishes all operations
spark_end = time.time()

# Print timing results:
# 1. How long loading the file took
# 2. How long the groupBy + sum took
# 3. Total duration of the entire Spark workflow
print(f"Spark Load Time: {load_done - spark_start:.2f} sec")
print(f"Spark GroupBy Time: {spark_end - load_done:.2f} sec")
print(f"Spark Total Time: {(spark_end - spark_start):.2f} sec")

# Shut down the Spark session to free resources
spark.stop()


--- Large Data Benchmark: Pandas vs PySpark (Itineraries Dataset) ---

--- PYSPARK TEST ---
Spark Load Time: 408.92 sec
Spark GroupBy Time: 394.96 sec
Spark Total Time: 803.88 sec


In [3]:
# ========================================================================
#                           PANDAS TEST
# ========================================================================
import pandas as pd
import time
import psutil
import os
# Name of the large dataset we want to benchmark
file = "itineraries.csv"

print("--- Large Data Benchmark: Pandas vs PySpark (Itineraries Dataset) ---")

print("\n--- PANDAS TEST ---")

# Track memory before Pandas
mem_before_pd = process.memory_info().rss / (1024 ** 2)
print(f"Memory before Pandas load: {mem_before_pd:.2f} MB")

# Start timer
pd_start = time.time()

# Load dataset in Pandas
df_pd = pd.read_csv(file)

# Time after loading
pd_load_done = time.time()

# Group the data just like Spark
pd_result = df_pd.groupby("startingAirport")["totalFare"].sum()

# End timing
pd_end = time.time()

# Memory after Pandas
mem_after_pd = process.memory_info().rss / (1024 ** 2)
print(f"Memory after Pandas operations: {mem_after_pd:.2f} MB")
print(f"Pandas memory increase: {mem_after_pd - mem_before_pd:.2f} MB")

# Print Pandas timings
print(f"Pandas Load Time: {pd_load_done - pd_start:.2f} sec")
print(f"Pandas GroupBy Time: {pd_end - pd_load_done:.2f} sec")
print(f"Pandas Total Time: {pd_end - pd_start:.2f} sec")


--- Large Data Benchmark: Pandas vs PySpark (Itineraries Dataset) ---

--- PANDAS TEST ---
Memory before Pandas load: 138.14 MB


MemoryError: Unable to allocate 256. KiB for an array with shape (32768,) and data type float64

In [1]:
import time
import psutil
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, col

# Name of the large dataset we want to benchmark
file = "itineraries.csv"

print("--- Large Data Benchmark: Pandas vs PySpark (Itineraries Dataset) ---")


# ========================================================================
#                           PYSPARK TEST
# ========================================================================
print("\n--- PYSPARK TEST ---")

# Track memory before anything happens
process = psutil.Process(os.getpid())
mem_before_spark = process.memory_info().rss / (1024 ** 2)  # MB
print(f"Memory before Spark load: {mem_before_spark:.2f} MB")

# Start timer for Spark
spark_start = time.time()

# Create a Spark session
spark = (
    SparkSession.builder
    .appName("ItinerariesBenchmark")
    .master("local[*]")
    .getOrCreate()
)

# Load CSV into Spark DataFrame
df_spark = spark.read.csv(file, header=True, inferSchema=True)

# Time checkpoint after loading
load_done = time.time()

# Group by airport and sum total fare
spark_result = (
    df_spark.groupBy(col("startingAirport"))
    .agg(sum("totalFare").alias("total_fare"))
)

# Force Spark to actually run (lazy execution)
spark_result.collect()

# End timing after groupBy
spark_end = time.time()

# Measure memory AFTER Spark operations
mem_after_spark = process.memory_info().rss / (1024 ** 2)  # MB

print(f"Memory after Spark operations: {mem_after_spark:.2f} MB")
print(f"Spark memory increase: {mem_after_spark - mem_before_spark:.2f} MB")

# Print Spark timings
print(f"Spark Load Time: {load_done - spark_start:.2f} sec")
print(f"Spark GroupBy Time: {spark_end - load_done:.2f} sec")
print(f"Spark Total Time: {(spark_end - spark_start):.2f} sec")

spark.stop()




--- Large Data Benchmark: Pandas vs PySpark (Itineraries Dataset) ---

--- PYSPARK TEST ---
Memory before Spark load: 93.83 MB
Memory after Spark operations: 129.34 MB
Spark memory increase: 35.50 MB
Spark Load Time: 894.47 sec
Spark GroupBy Time: 883.61 sec
Spark Total Time: 1778.08 sec
